# Getting started

### CLEF 2025 - CheckThat! Lab  - Task 1  Source Retrieval for Scientific Web Claims

This notebook enables participants of subtask 1 to quickly get started. It includes the following:
- Code to load data from HF Datasets, including:
    - code to load the collection set
    - code to load the query sets for training and development in three languages: English, German, and French
    - code to load the query sets for the final evaluation phase in three languages: English, German, and French
    
    **NOTE:** The data for Task 1 can be found in this Hugging Face repo: https://huggingface.co/datasets/sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims

    Please fill out the Declaration of Commitment on the Hugging Face repo to get access to the datasets.

- Code to run a baseline retrieval model (Semantic Similarity)
- Code to evaluate the baseline model
- Code to save predictions

Participants are free to use this notebook and add their own models for the competition.

# 0) Import libaraies, huggingface login, download embedding model

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

from torch import Tensor
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, pipeline, AutoModelForCausalLM, set_seed
from datasets import load_dataset
from tqdm import tqdm
from huggingface_hub import login as hf_login


os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["HF_TOKEN"] = "hf_TOKEN"

hf_login(token=os.getenv("HF_TOKEN"))
seed = 27

In [ ]:
# Default values from organizers's baseline code
embedding_model_name = 'intfloat/multilingual-e5-large'
langs = ["en", "de", "fr"]
batch_size = 128

In [ ]:
# Embedding model
embedder_tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
embedder_model = AutoModel.from_pretrained(embedding_model_name).cuda()

In [ ]:
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
set_seed(seed)

slm_pipeline = pipeline("text-generation", model="google/gemma-3-270m-it", device=f"cuda", dtype=torch.int8)

sample_chat_template =[
    [
        {
            "role": "system",
            "content": [{
                "type": "text",
                "text": """Rewrite abstracts into tweet-like summaries that captures all the core information (numbers, nouns, verbs)."""
            }],
        },
        {
            "role": "user",
            "content": [{
                "type": "text",
                "text": f"""'We have engineered the accumulation of provitamin D3 in tomato by genome editing, modifying a duplicated section of phytosterol biosynthesis in Solanaceous plants """}],
        },
    ],
]

output = slm_pipeline(sample_chat_template, max_new_tokens=50, temperature=0.0001, top_k=1)
print(output)
output[0][0]["generated_text"][-1]["content"]

# 1) Loading Datasets

### Load Collection Data

To load the dataset, follow the steps below:
1. Make sure you have access to the dataset by logging it to HuggingFace and requesting it using this link https://huggingface.co/datasets/sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims
2. On the HuggingFace website, create a "Read" Token as follows: Click on your profile picture on the top-right, then click on "Settings". A left menu appears, click on "Acces Tokens", then "Create New Token"
3. Connect this notebook to your HuggingFace account by running the notebook_login() cell below and entering your personal token you've just created.

Once these three steps are fulfilled, the load_dataset() function below works seamlessly.

In [ ]:
collection_data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "collection")["collection"]
collection_keys = np.array(collection_data["pubkey"])

# print(collection_data)
# print(collection_keys)
# collection_data[0]

In [ ]:
def rewrite_abstracts(
    texts,
    max_new_tokens=128,
):
    torch.cuda.empty_cache()
    messages = [
        [
            {
                "role": "system",
                "content": [{
                    "type": "text",
                    "text": "Rewrite the given abstract into a tweet-like summary that captures all the core information (numbers, nouns, verbs, claims, etc) concisely."
                }]
            },
            {
                "role": "user",
                "content": [{
                    "type": "text",
                    "text": text
                }]
            },
        ]
        for text in texts
    ]

    with torch.inference_mode():
        outputs = slm_pipeline(
            messages,
            batch_size=5,
            max_new_tokens=max_new_tokens,
            temperature=0.0001,
            top_k=1,
        )

    rewritten = [
        out[0]["generated_text"][-1]["content"].strip()
        for out in outputs
    ]

    return rewritten


def create_text_batch(batch):
    rewritten_abstracts = rewrite_abstracts(batch["abstract"])

    texts = [
        f"{title}\n{venue}\n{authors}\n{abstract}"
        for title, venue, authors, abstract in zip(
            batch["title"],
            batch["venue"],
            batch["authors"],
            rewritten_abstracts,
        )
    ]

    return {"text": texts}

collection_data = collection_data.map(
    create_text_batch,
    batched=True,
    batch_size=5,     # dataset batch size
)
# collection_data[0]

In [ ]:
def tokenize_with_padding(examples):
    return embedder_tokenizer(examples["text"], max_length=512, truncation=True, padding="max_length", return_tensors="pt")

collection_data = collection_data.map(tokenize_with_padding, batched=True, remove_columns=collection_data.column_names).with_format("torch", columns=["input_ids", "attention_mask"])

# print(collection_data[0]["input_ids"].shape)
# print(collection_data[0]["attention_mask"].shape)

## Load Query Data

In [ ]:
lang2train_data = {}
lang2dev_data = {}

for lang in langs:
  data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", lang)
  train = data["train"]
  dev = data["dev"]

  lang2train_data[lang] = train
  lang2dev_data[lang] = dev

In [ ]:
# print(lang2train_data["de"][0])
# print(lang2train_data["en"][0])
# print(lang2train_data["fr"][0])

# 2) Running Baseline

In [ ]:
def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

## Embed Collection Data

In [ ]:
embeddings = []
with torch.inference_mode():
    for batch in tqdm(DataLoader(collection_data, batch_size=batch_size, num_workers=10, prefetch_factor=2)):  # type: ignore
        batch = {k: v.to(embedder_model.device) for k, v in batch.items()}
        outputs = embedder_model(**batch)
        batch_embeddings = average_pool(outputs.last_hidden_state, batch['attention_mask'])
        embeddings.append(batch_embeddings.detach().cpu())

collection_embeddings = torch.vstack(embeddings)
collection_embeddings = F.normalize(collection_embeddings, p=2, dim=1)

In [ ]:
# torch.save(collection_embeddings, "/content/drive/MyDrive/collection_abstract_rewritten_embeddings.pt")
torch.save(collection_embeddings, "collection_abstract_rewritten_embeddings.pt")

## Preprocess Query Sets

In [ ]:
def process_text(example):
    text = example["text"]
    text = f'query: {text}'
    return {"text": text}

def tokenize_with_padding(examples):
    return embedder_tokenizer(examples["text"], max_length=128, truncation=True, padding="max_length", return_tensors="pt")

In [ ]:
lang2train_datasets = {}
lang2dev_datasets = {}

for lang in langs:
    print(f"Preprocess {lang} query sets")
    train = lang2train_data[lang]
    dev = lang2dev_data[lang]

    train = train.map(process_text)
    dev = dev.map(process_text)

    train = train.map(tokenize_with_padding, batched=True, remove_columns=train.column_names).with_format("torch", columns=["input_ids", "attention_mask"])
    dev = dev.map(tokenize_with_padding, batched=True, remove_columns=dev.column_names).with_format("torch", columns=["input_ids", "attention_mask"])

    lang2train_datasets[lang] = train
    lang2dev_datasets[lang] = dev

## Embed Query Sets

In [ ]:
lang2train_embeddings = {}
lang2dev_embeddings = {}

for lang in langs:
    print(f"Embed {lang} query sets")

    train = lang2train_datasets[lang]
    dev = lang2dev_datasets[lang]

    train_embeddings = []
    with torch.inference_mode():
        for batch in tqdm(DataLoader(train, batch_size=batch_size, num_workers=10, prefetch_factor=2)):
            batch = {k: v.to(embedder_model.device) for k, v in batch.items()}
            outputs = embedder_model(**batch)
            batch_embeddings = average_pool(outputs.last_hidden_state, batch['attention_mask'])
            train_embeddings.append(batch_embeddings.detach().cpu())

    train_embeddings = torch.vstack(train_embeddings)
    train_embeddings = F.normalize(train_embeddings, p=2, dim=1)

    dev_embeddings = []
    with torch.inference_mode():
        for batch in tqdm(DataLoader(dev, batch_size=batch_size, num_workers=10, prefetch_factor=2)):
            batch = {k: v.to(embedder_model.device) for k, v in batch.items()}
            outputs = embedder_model(**batch)
            batch_embeddings = average_pool(outputs.last_hidden_state, batch['attention_mask'])
            dev_embeddings.append(batch_embeddings.detach().cpu())

    dev_embeddings = torch.vstack(dev_embeddings)
    dev_embeddings = F.normalize(dev_embeddings, p=2, dim=1)

    lang2train_embeddings[lang] = train_embeddings
    lang2dev_embeddings[lang] = dev_embeddings

In [ ]:
torch.save({
    "lang2train_embeddings": lang2train_embeddings,
    "lang2dev_embeddings": lang2dev_embeddings
}, "query_embeddings.pt")

# 3) Run Baseline and Evaluate
You can run this without reruning everything from beginning

## Avoid rerun
- Define necessary function
- Load data/embedding

In [ ]:
# def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
#     last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
#     return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

# def process_text(example):
#     text = example["text"]
#     text = f'query: {text}'
#     return {"text": text}

# def tokenize_with_padding(examples):
#     return embedder_tokenizer(examples["text"], max_length=128, truncation=True, padding="max_length", return_tensors="pt")

# def create_text(example):
#     text = f'passage: {example["title"]}. {example["venue"]}. {example["abstract"]}. {example["authors"]}'
#     return {"text": text}

In [ ]:
# collection_data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "collection")["collection"]
# collection_data = collection_data.map(create_text)

# # Get collection keys for later use in evaluation as labels
# collection_keys = np.array(collection_data["pubkey"])

# Load pre-computed embeddings to save time if you have already run the embedding code once
collection_embeddings = torch.load("collection_abstract_rewritten_embeddings.pt", map_location="cpu")
query_embeddings = torch.load("query_embeddings.pt", map_location="cpu")

lang2train_embeddings = query_embeddings["lang2train_embeddings"]
lang2dev_embeddings = query_embeddings["lang2dev_embeddings"]


lang2train_data = {}
lang2dev_data = {}

for lang in langs:
  data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", lang)
  train = data["train"]
  dev = data["dev"]

  lang2train_data[lang] = train
  lang2dev_data[lang] = dev

In [ ]:
print(type(collection_embeddings), collection_embeddings.shape)
print(type(lang2train_embeddings["en"]), lang2train_embeddings["en"].shape)
print(type(lang2train_embeddings["de"]), lang2train_embeddings["de"].shape)
print(type(lang2train_embeddings["fr"]), lang2train_embeddings["fr"].shape)
print(type(lang2dev_embeddings["en"]), lang2dev_embeddings["en"].shape)
print(type(lang2dev_embeddings["de"]), lang2dev_embeddings["de"].shape)
print(type(lang2dev_embeddings["fr"]), lang2dev_embeddings["fr"].shape)
print(collection_keys[:5])
print(lang2train_data["en"][0])

## Calculate MRR@5 Score

Orgainizer's default code (without a reranker)

In [ ]:
def mrr_at_5(cos_sims, labels):

    # Get top-5 predictions
    top5_idxs = np.argsort(-cos_sims, axis=1)[:, :5]
    top5_keys = collection_keys[top5_idxs]

    # Compare labels to top-5 predictions
    labels_2d = labels[:, None]
    matches_mask = (top5_keys == labels_2d)

    # Get position of correct predictions
    positions = np.argmax(matches_mask, axis=1)
    positions[~matches_mask.any(axis=1)] = -1
    positions = positions.astype(int)

    # Get reciprocal rank
    rr = np.zeros_like(positions, dtype=float)
    mask = positions >= 0
    rr[mask] = 1.0 / (positions[mask] + 1)

    return float(rr.mean())

In [ ]:
for lang in langs:
  print(f"Evaluate {lang} on Train and Dev")
  train_embeddings = lang2train_embeddings[lang]
  dev_embeddings = lang2dev_embeddings[lang]

  train_cos_sims = (train_embeddings @ collection_embeddings.T).numpy()
  dev_cos_sims = (dev_embeddings @ collection_embeddings.T).numpy()

  train_labels = np.array([x["pubkey"] for x in lang2train_data[lang]])
  dev_labels = np.array([x["pubkey"] for x in lang2dev_data[lang]])

  print("MRR@5 Train:", mrr_at_5(train_cos_sims, train_labels))
  print("MRR@5 Dev:", mrr_at_5(dev_cos_sims, dev_labels), "\n")